In [1]:
# from trading_rookie.data.get_data import DataDownloader
# from trading_rookie.universe.sp500 import SP500Universe

# data_downloader = DataDownloader()
# builder = SP500Universe(
#     data_downloader,
#     start="2016-01-01",
#     end="2025-12-31",
#     min_observations=500,
# )

# universe50 = builder.build(universe_size=50, max_per_sector=8)

# print(universe50)
# print(builder.sector_distribution())

In [ ]:
import pandas as pd
from trading_rookie.data.get_data import DataDownloader
from trading_rookie.config.const import *
import time
import random
import numpy as np

series_list = []
data_downloader = DataDownloader()

def momentum_compatibility_score(price_series):
	ret = price_series.pct_change()
	mom = price_series / price_series.shift(63) - 1
	signal = (mom > 0).astype(int)
	strat_ret = signal.shift(1) * ret
	sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(252)
	cum = (1 + strat_ret).cumprod()
	peak = cum.cummax()
	dd = (cum / peak - 1).min()
	score = sharpe - 0.5 * abs(dd)
	return sharpe, dd, score

def build_ticker_sector_map(sector_map):
	ticker_to_sector = {}
	for sector, ticker_list in sector_map.items():
		for ticker in ticker_list:
			ticker_to_sector[ticker] = sector
	return ticker_to_sector

def select_universe50(score_df, ticker_to_sector, max_per_sector=8):
	universe = []	
	sector_count = {}
	for _, row in score_df.iterrows():
		ticker = row["ticker"]
		if ticker not in ticker_to_sector:
			continue
		sector = ticker_to_sector[ticker]
		if sector_count.get(sector, 0) < max_per_sector:
			universe.append(ticker)
			sector_count[sector] = sector_count.get(sector, 0) + 1
		if len(universe) == 50:
			break
	return universe

def download_close_matrix(tickers, start, end):
	close_df = pd.DataFrame()
	for t in tickers:
		try:
			time.sleep(random.randint(1, 3))
			df = data_downloader.yahoo(t, start, end)
		except Exception as e:
			print(f"Error downloading data for {t}: {e}")
			continue
		if len(df) < 500:
			continue
		s = df["Adj_close"].rename(t)
		series_list.append(s)
	close_df = pd.concat(series_list, axis=1)
	close_df = close_df.sort_index()
	close_df = close_df.dropna(axis=1, how="any") #ensure date aligned
	return close_df

sp500_tickers = [x for sector in SECTOR_MAP.values() for x in sector]

close_df = download_close_matrix(sp500_tickers, "2016-01-01", "2025-12-31")

scores = []

for ticker in close_df.columns:
	sharpe, dd, score = momentum_compatibility_score(close_df[ticker])
	scores.append((ticker, sharpe, dd, score))

score_df = pd.DataFrame(scores, columns=["ticker", "sharpe", "max_dd", "score"])
score_df = score_df.sort_values("score", ascending=False)

ticker_to_sector = build_ticker_sector_map(SECTOR_MAP)
universe50 = select_universe50(score_df, ticker_to_sector)

from collections import Counter

sector_distribution = Counter(ticker_to_sector[t] for t in universe50)

In [9]:
score_df.to_csv("outputs/score_df.csv", index=False)

In [ ]:
score = pd.read_csv("outputs/score_df.csv")

ticker_to_sector = build_ticker_sector_map(SECTOR_MAP)
universe50 = select_universe50(score, ticker_to_sector)

from collections import Counter

sector_distribution = Counter(ticker_to_sector[t] for t in universe50)

print(universe50)
print(sector_distribution)

['NVDA', 'APH', 'AXON', 'JPM', 'CAT', 'META', 'APO', 'AJG', 'WELL', 'NFLX', 'STX', 'WMT', 'AME', 'KKR', 'JCI', 'JBL', 'AAPL', 'SPGI', 'FIX', 'GS', 'PWR', 'PGR', 'AVGO', 'GRMN', 'BLDR', 'RJF', 'TT', 'CMG', 'MSI', 'TKO', 'LLY', 'AZO', 'TSLA', 'ALGN', 'NEM', 'BSX', 'ISRG', 'DECK', 'MPC', 'TMUS', 'HCA', 'NVR', 'COST', 'ABBV', 'TTWO', 'VTR', 'POOL', 'PKG', 'IRM', 'TMO']
Counter({'technology': 8, 'financials': 8, 'industrials': 8, 'consumer_discretionary': 7, 'healthcare': 7, 'communication_services': 4, 'real_estate': 3, 'consumer_staples': 2, 'materials': 2, 'energy': 1})


In [10]:
bad = []
for t in universe50:
    if t not in close_df.columns or close_df[t].dropna().shape[0] < 800:
        bad.append(t)
print("missing/short history:", bad)

pd.Series(universe50, name="ticker").to_csv("outputs/universe50.csv", index=False)

missing/short history: []
